**Objective**

Extract standardized image quality and geometric features from each dataset independently to create model-ready feature datasets.

Input

data/samples/
    ├── fairface/
    ├── celeba/
    ├── driver_drowsiness/
    └── driver_distraction/

Output

data/processed/

feature_fairface.csv
feature_celeba.csv
feature_driver_drowsiness.csv
feature_driver_distraction.csv

In [1]:
import cv2
print(cv2.__version__)

5.0.0


In [2]:
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image
from tqdm import tqdm

In [8]:
SAMPLES_DIR = Path("../data/samples")

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

FEATURE_FILE = PROCESSED_DIR / "feature_dataset.csv"

In [5]:
import cv2
import numpy as np

def extract_features(image_path):

    image = cv2.imread(str(image_path))

    if image is None:
        return None

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    height, width = gray.shape

    # Basic Image Statistics
    brightness = np.mean(gray)

    contrast = np.std(gray)

    blur_score = cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

    # File Properties
    size_mb = image_path.stat().st_size / (1024 * 1024)

    image_area = width * height

    aspect_ratio = width / height

    # Image Quality Metrics
    entropy = -np.sum(
        (hist := cv2.calcHist([gray],[0],None,[256],[0,256]) / gray.size)
        * np.log2(hist + 1e-10)
    )

    edge_density = (
        cv2.Canny(gray,100,200) > 0
    ).mean()

    return {

        "file_name": image_path.name,

        "width": width,

        "height": height,

        "image_area": image_area,

        "aspect_ratio": aspect_ratio,

        "size_mb": size_mb,

        "size_per_pixel": size_mb / image_area,

        "brightness": brightness,

        "contrast": contrast,

        "blur_score": blur_score,

        "entropy": entropy,

        "edge_density": edge_density,

        "is_square": int(abs(width-height) <= 10)

    }

In [9]:
metadata = pd.read_csv(
    "../data/processed/master_metadata.csv"
)

print(metadata.shape)
metadata.head()

(2400, 10)


,age,gender,race,service_test,file_name,dataset,Image_id,Left_eye_react,Right_eye_react,Label
0,4.0,0.0,3.0,False,fairface_00000.jpg,FairFace,NaN,NaN,NaN,NaN
1,1.0,1.0,5.0,False,fairface_00001.jpg,FairFace,NaN,NaN,NaN,NaN
2,2.0,1.0,0.0,False,fairface_00002.jpg,FairFace,NaN,NaN,NaN,NaN
3,3.0,1.0,0.0,False,fairface_00003.jpg,FairFace,NaN,NaN,NaN,NaN
4,2.0,0.0,5.0,True,fairface_00004.jpg,FairFace,NaN,NaN,NaN,NaN


In [17]:
feature_rows = []

for dataset_folder in sorted(SAMPLES_DIR.iterdir()):

    if not dataset_folder.is_dir():
        continue

    print(f"\nProcessing {dataset_folder.name}")

    metadata_file = dataset_folder / "metadata.csv"

    metadata_df = pd.read_csv(metadata_file)

    # Add dataset column
    metadata_df["dataset"] = dataset_folder.name

    for _, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):

        image_path = dataset_folder / "images" / row["file_name"]

        if not image_path.exists():
            continue

        features = extract_features(image_path)

        if features is None:
            continue

        feature_rows.append({

            **row.to_dict(),

            **features

        })

feature_dataset = pd.DataFrame(feature_rows)


Processing celeba


100%|██████████| 400/400 [00:00<00:00, 410.61it/s]



Processing driver_drowsiness


100%|██████████| 1000/1000 [00:10<00:00, 99.83it/s]



Processing fairface


100%|██████████| 1000/1000 [00:01<00:00, 590.80it/s]


In [18]:
feature_dataset.to_csv(
    "../data/processed/feature_dataset.csv",
    index=False
)

In [19]:
print(feature_dataset.info())

feature_dataset.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   file_name        2400 non-null   str    
 1   dataset          2400 non-null   str    
 2   width            2400 non-null   int64  
 3   height           2400 non-null   int64  
 4   image_area       2400 non-null   int64  
 5   aspect_ratio     2400 non-null   float64
 6   size_mb          2400 non-null   float64
 7   size_per_pixel   2400 non-null   float64
 8   brightness       2400 non-null   float64
 9   contrast         2400 non-null   float64
 10  blur_score       2400 non-null   float64
 11  entropy          2400 non-null   float32
 12  edge_density     2400 non-null   float64
 13  is_square        2400 non-null   int64  
 14  Image_id         1000 non-null   float64
 15  Left_eye_react   1000 non-null   str    
 16  Right_eye_react  1000 non-null   str    
 17  Label            1000 non

,file_name,dataset,width,height,image_area,aspect_ratio,size_mb,size_per_pixel,brightness,contrast,...,edge_density,is_square,Image_id,Left_eye_react,Right_eye_react,Label,age,gender,race,service_test
count,2400,2400,2400.000000,2400.00000,2400.000000,2400.000000,2400.000000,2.400000e+03,2400.000000,2400.000000,...,2400.000000,2400.000000,1000.000000,1000,1000,1000,1000.000000,1000.000000,1000.000000,1000
unique,2400,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,997,997,2,NaN,NaN,NaN,2
top,celeba_00000.jpg,driver_drowsiness,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,"[341.0, 118.0, 16.0, 4.0]","[177.0, 147.0, 27.0, 8.0]",closed_eyes,NaN,NaN,NaN,False
freq,1,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2,2,500,NaN,NaN,NaN,529
mean,NaN,NaN,336.333333,343.00000,136600.666667,0.969419,0.016748,1.255816e-07,101.435953,56.360213,...,0.042794,0.833333,67905.227000,NaN,NaN,NaN,3.484000,0.454000,2.897000,NaN
std,NaN,NaN,149.342499,142.87584,106198.134009,0.068396,0.013161,2.957871e-08,28.245679,14.007109,...,0.027572,0.372756,36583.300794,NaN,NaN,NaN,1.646346,0.498129,1.939111,NaN
min,NaN,NaN,178.000000,218.00000,38804.000000,0.816514,0.002681,5.342751e-08,23.628209,13.529063,...,0.000000,0.000000,127.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN
25%,NaN,NaN,224.000000,224.00000,50176.000000,1.000000,0.005569,1.055978e-07,82.935467,47.215571,...,0.022262,1.000000,37011.750000,NaN,NaN,NaN,3.000000,0.000000,1.000000,NaN
50%,NaN,NaN,224.000000,224.00000,50176.000000,1.000000,0.007198,1.219396e-07,99.400667,58.093719,...,0.040990,1.000000,71293.500000,NaN,NaN,NaN,3.000000,0.000000,3.000000,NaN
75%,NaN,NaN,512.000000,512.00000,262144.000000,1.000000,0.030799,1.400664e-07,118.123814,65.708902,...,0.058334,1.000000,99313.250000,NaN,NaN,NaN,4.000000,1.000000,5.000000,NaN
